In [1]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import netket as nk
import netket.experimental as nkx
import sys
sys.path.append('..')
from NES_VMC import NESTotalAnsatz, create_machine,\
    SingleStateAnsatz,create_single_machine,\
    create_machine_matrix,Ham_psi,Ham_Psi,NES_loss_energy,nes_vmc_gradient,\
    NESFermionHopRule,compute_qgt,sampler_info,\
    create_machine_matrix_stable,create_single_machine_gauge_fixed,create_machine_max_stable,\
    NESTotalAnsatz_stable,create_machine_stable,NES_loss_energy_stable,nes_vmc_gradient_stable,\
    make_grad_fn,make_qgt_fn
import optax
from scipy.sparse.linalg import eigsh
from typing import Callable
from functools import partial
from jax.flatten_util import ravel_pytree
import time
import itertools
from pyscf import gto, scf, fci
import numpy as np



jnp.set_printoptions(
    linewidth=9999,
    threshold=jnp.inf,
    precision=8,
    suppress=False,
)


/opt/miniconda3/envs/Netket/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: uv is a replacement for pip which helps you follow good software practices.

In [2]:
from LiH import SINGLE_SIZE,ha,hi,hi_ext,ext_edges,single_edges_full,K,Hatree_Fock

LiH 分子基本信息
HF energy = -7.86245187 Ha
Total electrons = (2, 2)
Total basis functions = 6

Total orbitals in STO-3G: 6

LiH / STO-3G 基准能量
E0 = -7.88259092 Ha | excitation = 0.0000 eV
E1 = -7.76563664 Ha | excitation = 3.1825 eV
E2 = -7.74858736 Ha | excitation = 3.6464 eV
E3 = -7.71601970 Ha | excitation = 4.5326 eV
E4 = -7.71601970 Ha | excitation = 4.5326 eV
E5 = -7.69619638 Ha | excitation = 5.0721 eV

Active Space Configuration:
  n_orbitals = 4
  n_alpha = 2, n_beta = 2
  Total electrons = (2, 2)

Hilbert 信息
K = 4
hi.size = 8
hi_ext.size = 32
SINGLE_SIZE = 8
target_loss = sum(E_fcis[:K]) = -31.11283462

HF reference state: [0 0 1 1 0 0 1 1]
Alpha orbitals: [0, 1, 2, 3]... (4 total)
Beta orbitals: [4, 5, 6, 7]... (4 total)

Single edges (total): 12


In [ ]:
nes_rule = NESFermionHopRule(
    edges=ext_edges,
    K=K,
    single_size=SINGLE_SIZE,
)

In [6]:
from NES_VMC import (
    NESTotalAnsatz_stable,
    NESTotalAnsatz_gauge_stable,
    create_gauge_fixed_total_machines,
    create_single_machine_gauge_fixed,
    make_grad_fn,
    make_qgt_fn,
    create_machine_gauge_synthesis,
    create_machine_gauge_stable,
    create_single_machine_gauge_fixed,
)

# ====================== 模型初始化 ======================

total_ansatz = NESTotalAnsatz_gauge_stable(SINGLE_SIZE, K, 
                                           SINGLE_SIZE+K, ref_state=Hatree_Fock,
                                           rngs=nnx.Rngs(11))

total_machine_log_Psi_gauge, graphdef, total_params =  create_machine_gauge_stable(total_ansatz)
#total_params = history['params'][-1]

total_machine_log_Psi_gauge, \
total_machine_L_stable, \
total_machine_shift, \
total_machine_L_gauge = create_machine_gauge_synthesis(total_ansatz)

In [7]:
import jax
import jax.numpy as jnp
import jax.lax as lax

def f(x):
    feat = x * 2
    feat_detach = lax.stop_gradient(feat) # 梯度截断点
    return feat_detach + 3

# 生成jaxpr计算图文本
print(jax.make_jaxpr(f)(jnp.array(1.0)))

{ lambda ; a:f64[]. let
    b:f64[] = mul a 2.0:f64[]
    c:f64[] = stop_gradient b
    d:f64[] = add c 3.0:f64[]
  in (d,) }


In [8]:
!pip install jaxprvis -q

/opt/miniconda3/envs/Netket/lib/python3.11/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()
